# Lab 4: Ecology Models and Snow Leopard Tracking in Python
**Course:** Environmental Science for Sustainable Development  
**Source:** Ecology Lab, S. Hsiang  
**Format:** Python Jupyter notebook

This notebook follows the assignment in order. Sections 1 and 2 contain Python code; Section 3 retains empty code cells. Written answers and interpretations will be added after the coding sections.

## Setup and submission checklist
- Write the scripts yourself in Python and include the code in the submission.
- Read each simulation section before implementing it. Plan inputs, outputs, array dimensions, indexing, and information flow; include the requested planning diagrams.
- For each simulation, include its plan, code, plots, and written interpretation. Title plots and label axes.
- State any chosen initial values and modeling assumptions. Complete the remaining empty code cells section by section.

# 1. Solving difficult equations
## 1.1 Plot the two surfaces
Create a two-dimensional grid from -5 to 5 in both x and y, spaced by 0.1. Plot the population surface z = x² + sin(y) + xy. On the same axes, plot the conservation surface z = x + y. Describe their shapes and where they appear to meet.

**Python implementation:** grid construction and surface plots.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

x_values = np.arange(-5.0, 5.0 + 0.05, 0.1)
y_values = np.arange(-5.0, 5.0 + 0.05, 0.1)
X, Y = np.meshgrid(x_values, y_values)

z_population = X**2 + np.sin(Y) + X * Y
z_conservation = X + Y

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(projection="3d")
ax.plot_surface(X, Y, z_population, color="tab:blue", alpha=0.65,
                rstride=3, cstride=3)
ax.plot_surface(X, Y, z_conservation, color="tab:orange", alpha=0.65,
                rstride=3, cstride=3)
ax.set(xlabel="x", ylabel="y", zlabel="z",
       title="Population and conservation surfaces")
ax.legend(handles=[Patch(color="tab:blue", label="Population model"),
                   Patch(color="tab:orange", label="Conservation equation")])
ax.view_init(elev=25, azim=-55)
plt.tight_layout()
plt.show()

## 1.2 Find and interpret the intersection
Plot the intersection of the surfaces. At x = 1, how many y values satisfy both equations? What is z at each solution? Explain how the plot and numerical estimates support the answer.

**Python implementation:** intersection plot and x = 1 calculations.

In [ ]:
from scipy.optimize import brentq

# The zero contour of this difference is the intersection in the x-y plane.
difference = z_population - z_conservation
x_fixed = 1.0

def difference_at_fixed_x(y):
    return x_fixed**2 + np.sin(y) + x_fixed * y - (x_fixed + y)

y_scan = np.linspace(-5.0, 5.0, 2001)
sample_values = difference_at_fixed_x(y_scan)
roots = []
for i in range(len(y_scan) - 1):
    if np.isclose(sample_values[i], 0.0, atol=1e-12):
        roots.append(y_scan[i])
    elif sample_values[i] * sample_values[i + 1] < 0:
        roots.append(brentq(difference_at_fixed_x, y_scan[i], y_scan[i + 1]))
if np.isclose(sample_values[-1], 0.0, atol=1e-12):
    roots.append(y_scan[-1])

y_roots = np.array(sorted(set(np.round(roots, 10))))
z_roots = x_fixed + y_roots

fig, ax = plt.subplots(figsize=(8, 6))
ax.contour(X, Y, difference, levels=[0], colors="tab:blue")
ax.plot([], [], color="tab:blue", label="Surface intersection")
ax.axvline(x_fixed, color="tab:orange", linestyle="--", label="x = 1")
ax.scatter(np.full_like(y_roots, x_fixed), y_roots, color="black",
           zorder=3, label="Solutions at x = 1")
ax.set(xlabel="x", ylabel="y", xlim=(-5, 5), ylim=(-5, 5),
       title="Intersection of the two surfaces")
ax.set_aspect("equal")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Number of solutions at x = {x_fixed:g}: {len(y_roots)}")
for y_value, z_value in zip(y_roots, z_roots):
    print(f"y = {y_value:.6f}, z = {z_value:.6f}")

# 2. Koala population models
## 2.1 Unrestricted growth: assumptions and plan
Given one-third mature males, one-third mature females, one-third juveniles, and a one-half annual birth probability per mature female, what is the growth parameter r in N(t+1) = N(t) + rN(t)? What extra information is needed to find population at a specified time? Choose and report an initial population μ.

Before coding, draw the input and output arrays, their dimensions and indices, and the yearly calculation flow. Attach or reproduce the plan here.

**Heuristic array and computation plan (Python indices):**



The stored time and population arrays each have shape (T+1,). Expected births are a temporary value in each yearly update. The initial population and numerical response can be filled in when Section 2 is solved.

## 2.2 Simulate unrestricted growth
Use one-year time steps to compute population over time from μ. Plot population against time and describe its behavior. Include the script and the plot.

**Python implementation:** growth simulation and plot.

In [ ]:
female_fraction = 1 / 3
birth_probability = 1 / 2
r_koala = female_fraction * birth_probability
mu = 120.0  # Chosen initial population
years = 80
time = np.arange(years + 1)
population = np.empty(years + 1, dtype=float)
population[0] = mu
for t in range(years):
    expected_births = r_koala * population[t]
    population[t + 1] = population[t] + expected_births
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(time, population)
ax.set(xlabel="Year", ylabel="Koalas", title="Unrestricted koala growth")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f"r = {r_koala:.6f}; N0 = {mu:g}")


## 2.3 Carrying capacity: algebra and plan
A fixed forest has 5,000 eucalyptus trees; each koala needs five trees. Determine koala carrying capacity K. For the discrete logistic update N(t+1) = N(t) + rN(t)[1 − N(t)/K], check algebraically the stated growth or decline when N is below or above K, and the small-population limit.

Plan the arrays and update order before coding. Attach or reproduce the requested plan.

**Derivation / plan:** To be completed.

## 2.4 Characterize discrete logistic dynamics
Simulate several initial populations. How many types of behavior arise? Are there stable, indefinitely oscillating, or chaotic solutions? What in the model allows oscillations, and are they a numerical artifact or a plausible feature of nature? Use at most two plots, with multiple runs per plot if useful.

**Python implementation:** initial-condition experiments; at most two plots.

In [ ]:
eucalyptus_trees = 5000
trees_per_koala = 5
K_koala = eucalyptus_trees / trees_per_koala

def simulate_logistic(initial, rate, capacity, years):
    # Keep the PDF's update exact; stop if a negative count would occur.
    values = np.full(years + 1, np.nan)
    values[0] = initial
    for t in range(years):
        n = values[t]
        next_n = n + rate * n * (1 - n / capacity)
        if not np.isfinite(next_n) or next_n < 0:
            return values, t + 1
        values[t + 1] = next_n
    return values, None

years_logistic = 100
time_logistic = np.arange(years_logistic + 1)
initial_koalas = [50, 250, 800, 1000, 1500]
fig, ax = plt.subplots(figsize=(9, 5))
for n0 in initial_koalas:
    trajectory, invalid_year = simulate_logistic(n0, r_koala, K_koala, years_logistic)
    ax.plot(time_logistic, trajectory, label=f"N0 = {n0}")
    print(f"Koalas N0={n0}: first invalid year = {invalid_year}")
ax.axhline(K_koala, color="black", linestyle="--", label="Carrying capacity")
ax.set(xlabel="Year", ylabel="Koalas", title="Koala logistic model")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f"Koala carrying capacity = {K_koala:g}")


## 2.5 Beetles without koalas
Beetles live one year; half are female and each female produces ten offspring. Two hundred beetles can live on one eucalyptus tree. Find the beetle growth parameter and carrying capacity for the same 5,000-tree forest. Use the logistic equation to characterize beetles alone. Are the solutions qualitatively different from the koala model? Why?

**Plan / response:** To be completed.

**Python implementation:** beetle-only simulations and plot.

In [ ]:
female_beetle_fraction = 1 / 2
offspring_per_female = 10
# Adults die after reproducing; the young replace the current adults.
r_beetle = female_beetle_fraction * offspring_per_female - 1
beetles_per_tree = 200
K_beetle = eucalyptus_trees * beetles_per_tree
years_beetle = 25
time_beetle = np.arange(years_beetle + 1)
initial_beetles = [1, 100, 1000, 100_000, 1_000_000]
fig, ax = plt.subplots(figsize=(9, 5))
for b0 in initial_beetles:
    trajectory, invalid_year = simulate_logistic(b0, r_beetle, K_beetle, years_beetle)
    ax.plot(time_beetle, trajectory / K_beetle, label=f"B0 = {b0:,}")
    print(f"Beetles B0={b0:,}: first invalid year = {invalid_year}")
ax.axhline(1, color="black", linestyle="--", label="Carrying capacity")
ax.set(xlabel="Year", ylabel="Beetles / carrying capacity",
       title="Beetle logistic model without koalas")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f"Beetle r = {r_beetle:g}; carrying capacity = {K_beetle:,.0f}")


## 2.6 Koala–beetle competition: equations and plan
Use each species' logistic equation with a competitor term, converting the competitor population into equivalent consumers of eucalyptus. What are the two conversion constants, and what are the two coupled update equations? Explain why the constants differ.

Plan the two-population arrays and simultaneous yearly update. Attach or reproduce the requested plan.

**Equations / plan:** To be completed.

## 2.7 Competition outcomes
Introduce a small beetle population (initial beetles no more than 100) at year zero. Simulate a variety of initial koala populations. How many distinct long-run equilibria occur? Show the relevant trajectories and discuss the outcomes.

**Python implementation:** coupled simulations and comparison plots.

In [ ]:
# Convert beetle consumption to koala units and vice versa.
beta_koala = (1 / beetles_per_tree) / trees_per_koala
beta_beetle = trees_per_koala * beetles_per_tree

def simulate_competition(koalas_initial, beetles_initial, years):
    koalas = np.full(years + 1, np.nan)
    beetles = np.full(years + 1, np.nan)
    koalas[0], beetles[0] = koalas_initial, beetles_initial
    for t in range(years):
        k, b = koalas[t], beetles[t]
        # Simultaneous updates use only year-t values.
        k_next = k + r_koala * k * (1 - (k + beta_koala * b) / K_koala)
        b_next = b + r_beetle * b * (1 - (b + beta_beetle * k) / K_beetle)
        if not np.all(np.isfinite([k_next, b_next])) or min(k_next, b_next) < 0:
            return koalas, beetles, t + 1
        koalas[t + 1], beetles[t + 1] = k_next, b_next
    return koalas, beetles, None

years_competition = 60
time_competition = np.arange(years_competition + 1)
initial_koalas_competition = [100, 500, 1000, 1500]
initial_beetles_competition = 50  # Introduced at year 0; no more than 100.
fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
for k0 in initial_koalas_competition:
    koalas, beetles, invalid_year = simulate_competition(
        k0, initial_beetles_competition, years_competition)
    axes[0].plot(time_competition, koalas, label=f"K0 = {k0}")
    axes[1].plot(time_competition, beetles, label=f"K0 = {k0}")
    print(f"Competition K0={k0}: first invalid year = {invalid_year}")
axes[0].set(ylabel="Koalas", title="Koala-beetle competition")
axes[1].set(xlabel="Year", ylabel="Beetles")
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f"beta_koala = {beta_koala:g}; beta_beetle = {beta_beetle:g}")


## 2.8 Koala–snake predator–prey model: equations and plan
Replace beetles with tree snakes. Each snake eats one koala per month. Modify the koala equation for annual predation. One-quarter of snakes are mature females, each reproducing successfully in a year with probability one-eighth; formulate the snake update analogous to the logistic equation, with snake carrying capacity depending on koala population. State any additional assumptions.

Plan the population arrays and yearly calculation flow. Attach or reproduce the requested plan.

**Equations / plan:** To be completed.

## 2.9 Predator–prey outcomes and thresholds
Start koalas at their carrying capacity. Vary the number of snakes introduced at year zero. How does the system respond? Are there introduction thresholds where a small change in initial snakes causes a large change in the long-run outcome? Plot and discuss the evidence.

**Python implementation:** predator–prey simulations and plots.

In [ ]:
female_snake_fraction = 1 / 4
snake_birth_probability = 1 / 8
r_snake = female_snake_fraction * snake_birth_probability
koalas_eaten_per_snake_year = 12

def simulate_predator_prey(snakes_initial, years):
    koalas = np.full(years + 1, np.nan)
    snakes = np.full(years + 1, np.nan)
    koalas[0], snakes[0] = K_koala, snakes_initial
    for t in range(years):
        k, s = koalas[t], snakes[t]
        snake_capacity = k / koalas_eaten_per_snake_year
        if snake_capacity == 0 and s > 0:
            return koalas, snakes, t + 1  # Snake equation is undefined.
        k_next = k + r_koala * k * (1 - k / K_koala) - 12 * s
        s_next = (s + r_snake * s * (1 - s / snake_capacity)
                  if snake_capacity > 0 else 0.0)
        if not np.all(np.isfinite([k_next, s_next])) or min(k_next, s_next) < 0:
            return koalas, snakes, t + 1
        koalas[t + 1], snakes[t + 1] = k_next, s_next
    return koalas, snakes, None

years_predator = 150
time_predator = np.arange(years_predator + 1)
initial_snakes = [0, 1, 2, 5, 10, 20, 40, 80]
fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
for s0 in initial_snakes:
    koalas, snakes, invalid_year = simulate_predator_prey(s0, years_predator)
    axes[0].plot(time_predator, koalas, label=f"S0 = {s0}")
    axes[1].plot(time_predator, snakes, label=f"S0 = {s0}")
    print(f"Predator-prey S0={s0}: first invalid year = {invalid_year}")
axes[0].set(ylabel="Koalas", title="Koala-snake predator-prey model")
axes[1].set(xlabel="Year", ylabel="Snakes")
for ax in axes:
    ax.legend(ncol=2, fontsize=8)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Scan introductions one snake at a time to locate abrupt changes.
snake_grid = np.arange(0, 101)
first_invalid = [
    simulate_predator_prey(int(s0), years_predator)[2] for s0 in snake_grid
]
invalid_plot = [years_predator + 1 if y is None else y for y in first_invalid]
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(snake_grid, invalid_plot, ".-")
ax.set(xlabel="Snakes introduced at year 0",
       ylabel="First invalid year (151 means none)",
       title="Sensitivity to initial snake population")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 2.10 Don’t compute, just think
Would an introduced snake prefer beetles to be introduced? Would a beetle prefer snakes to be introduced? Discuss both questions briefly, using the food-web relationships rather than another computation.

**Response:** To be completed.

# 3. Tracking snow leopards
## 3.1 Movement model: design and short check
Model a 10 × 10 grid with daily steps. A leopard must move to one of the adjacent eight cells that remains inside the habitat, choosing uniformly among available cells. Plan position and time storage, boundary handling, random movement, and reusable functions; include a diagram of dimensions, indices, and decision order. Test each part separately.

Run and inspect a single leopard for fewer than ten steps, then produce the requested ten-day script output.

**Plan:** To be completed.

**Python placeholder:** single-leopard movement and short-run checks.

## 3.2 One leopard over a year
Run the checked movement model for 365 days. Plot its path through the 10 × 10 region and check that the path behaves as expected.

**Python placeholder:** year-long single path and plot.

## 3.3 Multiple leopards
Generalize to a parameterized population N; multiple animals may occupy one cell. Check N = 2 before larger populations. Plot the 365-day paths for N > 2 with a different color for each animal.

**Python placeholder:** multiple-leopard movement and paths.

## 3.4 Camera-trap verification
Place one camera at cell (3,3). Record one if at least one leopard occupies that cell on a day, otherwise zero; multiple leopards on one day still count as one photo. Show two subplots: animal paths with the camera marked by a star, and the daily zero-or-one observation series. Confirm the number of recorded observations agrees with the plotted paths. State clearly how PDF cell labels map to Python array indices.

**Python placeholder:** camera observations, comparison, and two-panel plot.

## 3.5 Monte Carlo estimate of population
For a 365-day camera study at (3,3) with seven observed photos, estimate the most likely population N. For each fixed N, repeat the simulation 1,000 times and record the total sightings M per run. During repeated runs, store only what is needed to count sightings; keep full paths for the earlier validation plots.

Start at N = 1 and continue through N = 5; extend to larger N if the greatest estimated probability of M = 7 occurs at the boundary. Plot the sighting distributions as subplots. What is the estimated probability of exactly seven sightings for each N, and which N maximizes it? State simulation uncertainty and the final estimate.

**Python placeholder:** repeated simulations, distributions, likelihood comparison, and plots.

## 3.6 Choose a camera location
With N = 3 and 1,000 year-long runs per location, compare cameras at (3,3), (1,1), and (5,5). Reuse the (3,3) results if applicable. Compare the variance of each location's sighting-count distribution. Which has the smallest and largest variance? What does this mean, and where would you recommend placing the camera before the field experiment?

**Python placeholder:** location comparison, distributions, variances, and plot.

## 3.7 Extra credit: precision of the population estimate
Explain why the variance of sightings M at a location differs from the variance of the population estimator N̂ based on maximum likelihood. Which criterion better measures precision of the population estimate? Describe how to estimate the variance of N̂ across camera locations.

**Response:** To be completed.

**Optional Python placeholder (extra extra credit):** estimate and compare those estimator variances.

# Summary of results
When the analysis is complete, summarize the surface intersection, population-model behaviors, snow-leopard population estimate, and camera-location recommendation. Separate computed findings from modeling assumptions and simulation uncertainty.

**Summary:** To be completed.